In [ ]:
import os
import json
import numpy as np
import pandas as pd

folder_path = r"D:\OneDrive\Trading\Market Making\data\runs\run_20260530_142928"

snapshots = pd.read_parquet(os.path.join(folder_path, "snapshots.parquet"))
trades = pd.read_parquet(os.path.join(folder_path, "trades.parquet"))
quotes = pd.read_parquet(os.path.join(folder_path, "quotes.parquet"))
fills = pd.read_parquet(os.path.join(folder_path, "fills.parquet"))

events = pd.read_parquet(os.path.join(folder_path, "events.parquet"))
with open(os.path.join(folder_path, "orderbook_snapshot.json"), "r") as f:
    orderbook_snapshot = json.load(f)

In [8]:
snapshots

,ts,symbol,best_bid,best_ask,mid,microprice,best_bid_tick,best_ask_tick,mid_tick,spread,...,my_bid_tick,my_ask_tick,bid_distance_touch,ask_distance_touch,bid_distance_spread,ask_distance_spread,bid_delta,ask_delta,quote_churn,future_return
0,1780235227114,BTCUSDT,73900.32,73900.33,73900.325,73900.325922,7390032,7390033,7390033,0.01,...,7390032,7390036,0.0,0.03,-0.01,0.04,0.0,0.0,0.0,0.0
1,1780235227214,BTCUSDT,73900.32,73900.33,73900.325,73900.325922,7390032,7390033,7390033,0.01,...,7390032,7390036,0.0,0.03,-0.01,0.04,0.0,0.0,0.0,0.0
2,1780235227314,BTCUSDT,73900.32,73900.33,73900.325,73900.325922,7390032,7390033,7390033,0.01,...,7390032,7390036,0.0,0.03,-0.01,0.04,0.0,0.0,0.0,0.0
3,1780235227414,BTCUSDT,73900.32,73900.33,73900.325,73900.325922,7390032,7390033,7390033,0.01,...,7390032,7390036,0.0,0.03,-0.01,0.04,0.0,0.0,0.0,0.0
4,1780235227514,BTCUSDT,73900.32,73900.33,73900.325,73900.325922,7390032,7390033,7390033,0.01,...,7390032,7390036,0.0,0.03,-0.01,0.04,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131,1780235240214,BTCUSDT,73900.32,73900.33,73900.325,73900.325167,7390032,7390033,7390033,0.01,...,7390032,7390035,0.0,0.02,-0.01,0.03,0.0,0.0,0.0,NaN
132,1780235240316,BTCUSDT,73900.32,73900.33,73900.325,73900.325167,7390032,7390033,7390033,0.01,...,7390032,7390035,0.0,0.02,-0.01,0.03,0.0,0.0,0.0,NaN
133,1780235240414,BTCUSDT,73900.32,73900.33,73900.325,73900.325167,7390032,7390033,7390033,0.01,...,7390032,7390035,0.0,0.02,-0.01,0.03,0.0,0.0,0.0,NaN
134,1780235240515,BTCUSDT,73900.32,73900.33,73900.325,73900.325167,7390032,7390033,7390033,0.01,...,7390032,7390035,0.0,0.02,-0.01,0.03,0.0,0.0,0.0,NaN


In [ ]:
"""
Models corrects microprice based on predicted future price movement (drift) to maximise E[PnL] / Utility / Min Adverse Selection due to symmetrical quoting
"""

In [ ]:
"""
Research Hypothesis (core version)

Microprice is a near-term unbiased estimator of mid-price, but its forecasting error over short horizons is conditionally predictable from limit order book state and trade flow features.




Let microprice be the baseline estimator of short-horizon fair value. We hypothesize that the error

y_t = log(mid_{t+h}) - log(microprice_t)

is not white noise, but has a non-zero conditional expectation given market microstructure features:

E[y_t | mid X_t] where X_t includes order book imbalance, trade imbalance, volatility, inventory state, and queue dynamics.


Trading interpretation (what your engine is really testing)

Predictable deviations of future mid-price from current microprice can be exploited to adjust market-making reservation prices, improving expected PnL relative to a baseline microprice-centered quoting strategy.


System-level hypothesis (your full pipeline)

A market-making strategy that adjusts quote centers using a learned estimate of microprice forecasting error (conditional drift) achieves higher risk-adjusted returns than a baseline strategy that assumes microprice is an unbiased short-horizon estimator of future mid-price.

One-line intuition (what you can say in interview)

“I’m testing whether microprice is only locally efficient on average, but conditionally predictable using order book and trade flow signals—and whether that predictability can improve market-making quotes.”
"""

In [2]:
snapshots

,ts,symbol,best_bid,best_ask,mid,microprice,best_bid_tick,best_ask_tick,mid_tick,spread,...,my_bid_tick,my_ask_tick,bid_distance_touch,ask_distance_touch,bid_distance_spread,ask_distance_spread,bid_delta,ask_delta,quote_churn,future_return
0,1780053744914,BTCUSDT,73687.99,73688.00,73687.995,73687.991494,7368799,7368800,7368799,0.01,...,7368793,7368800,-0.06,0.00,-0.07,0.01,0.0,0.0,0.0,0.0
1,1780053745014,BTCUSDT,73687.99,73688.00,73687.995,73687.991494,7368799,7368800,7368799,0.01,...,7368793,7368800,-0.06,0.00,-0.07,0.01,0.0,0.0,0.0,0.0
2,1780053745114,BTCUSDT,73687.99,73688.00,73687.995,73687.991494,7368799,7368800,7368799,0.01,...,7368793,7368800,-0.06,0.00,-0.07,0.01,0.0,0.0,0.0,0.0
3,1780053745214,BTCUSDT,73687.99,73688.00,73687.995,73687.991494,7368799,7368800,7368799,0.01,...,7368793,7368800,-0.06,0.00,-0.07,0.01,0.0,0.0,0.0,0.0
4,1780053745314,BTCUSDT,73687.99,73688.00,73687.995,73687.991494,7368799,7368800,7368799,0.01,...,7368793,7368800,-0.06,0.00,-0.07,0.01,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7108,1780054455714,BTCUSDT,73560.00,73560.01,73560.005,73560.009979,7356000,7356001,7356000,0.01,...,7356000,7356009,0.00,0.08,-0.01,0.09,0.0,0.0,0.0,NaN
7109,1780054455814,BTCUSDT,73560.00,73560.01,73560.005,73560.009979,7356000,7356001,7356000,0.01,...,7356000,7356009,0.00,0.08,-0.01,0.09,0.0,0.0,0.0,NaN
7110,1780054455914,BTCUSDT,73560.00,73560.01,73560.005,73560.009979,7356000,7356001,7356000,0.01,...,7356000,7356009,0.00,0.08,-0.01,0.09,0.0,0.0,0.0,NaN
7111,1780054456015,BTCUSDT,73560.00,73560.01,73560.005,73560.009979,7356000,7356001,7356000,0.01,...,7356000,7356009,0.00,0.08,-0.01,0.09,0.0,0.0,0.0,NaN


In [ ]:
"""
1. What you are actually assuming without the model

It's not:

“price will be at microprice in the next h intervals”

That would be too strong.

Instead, your implicit baseline is:

microprice(t) is your best instantaneous estimate of fair value at time t

So the null hypothesis you are testing is:

“microprice(t) is an unbiased / efficient predictor of future mid-price (at horizon h)”

2. What your y is doing in that context

You defined: y_t = log(mid_t+h) - log(microprice_t) (y is the realized short-term drift of mid-price relative to current microprice)

So yes:

You are measuring the forecast error of microprice as a predictor of future mid-price

But more precisely:

You are measuring how future mid deviates from current microprice over horizon h

4. Clean statistical interpretation

Your setup is equivalent to:

Baseline model:

p^_t+h = microprice_t

Error: error_t = mid_t + h-microprice_t

Then you learn:
E[error_t | features_t]

So yes, but more precisely:

You are modeling the conditional bias of microprice as a predictor of future mid

1. Your baseline belief

p^_t+h = microprice_t

You are implicitly saying:

microprice(t) is my best zero-model estimate of future mid-price

2. What your model is testing:

 y_t = log(mid_t+h) - log(microprice_t)

when microprice is systematically wrong, and by how much

That means your improved predictor becomes:

p^_t+h = microprice_t + E[y_t | features_t] (y_t is the short-term drift of mid-price relative to current microprice)

3. So can your model do better?
Yes — but only if this is true:

The error of microprice is predictable from current market features

That is the entire alpha.

If:

microprice errors are random → model cannot improve
microprice errors are structured → model can extract signal
"""

In [ ]:
"""
h = 10

“Predict where the market will be 10 snapshots into the future.”

Don't pick one h blindly.

Instead:

You should measure it

Compute:

correlation(y, model output proxy) across different h or even simpler:

IC(h) = Corr(features_t,mid_t+ h-microprice_t)

Then plot:

IC vs horizon

This gives you the true optimal horizon of your signal

7. What I would expect for your setup

Given:

BTCUSDT
order book features
microprice baseline
100ms updates

You will likely see:

weak at 100ms
peaks around 300ms-2s
decays after ~5-10s

Compute future return

Model A → 1s prediction (microstructure edge)
Model B → 5s prediction (directional drift)

"""

df = snapshots

TICK_SIZE = 0.01
"""
single time horizon

Use ONLY 1s horizon first

train stability
validate signal exists
check PnL attribution

"""
# h = 10 # 1000ms horizon

h = 2 # try to lower horizon to 200ms instead of 1000ms since number of rows is too small

df = df.sort_values("ts").reset_index(drop=True)


# horizon = pd.Timedelta(milliseconds=500) # closest future timestamp ≥ t + horizon
# future = df[["ts", "mid"]].copy()
# future = future.rename(columns={"mid": "future_mid"})
# future["ts"] = future["ts"] - horizon
# df = pd.merge_asof(
#     df.sort_values("ts"),
#     future.sort_values("ts"),
#     on="ts",
#     direction="forward",
#     suffixes=("", "_future")
# )

df["future_mid"] = df["mid"].shift(-h) # shift h into the future

# df["y"] = np.log(df["future_return_500ms"]) - np.log(df["microprice"])
#df["y"] = np.log(df["future_mid"]) - np.log(df["microprice"]) # “Does microprice at time t predict future mid move?”, log space drift, log returns
df["y"] = (df["future_mid"] - df["microprice"]) / df["microprice"] # price space drift, simple returns, more useful for ml model



"""
Interpretation:

You are NOT predicting:

instantaneous next tick return ❌
pure future mid price ❌

You ARE predicting:

“Where will the average future mid-price over the next h steps be, relative to today’s microprice?”

So your model learns a smoothed forward-looking price drift signal.

2. Your model IS actually:

You are training: y_t = f(market state_t)

where: y_t = log(future mid) - log(microprice_t)

So rearranging:

log(future mid) = log(microprice_t) + f(features_t)

>>> “Expected future drift relative to fair price now”

X = [
    microprice,
    spread,
    order_imbalance,
    trade_imbalance,
    inventory,
    volatility,
    queue_ahead_bid,
    queue_ahead_ask
]

y = future_mid - microprice (log space)

So the model is learning:

“Given the current market state (including microprice), how far will price move away from today’s fair price estimate?

"""

# df["y"] = np.log(df["future_mid"]) - np.log(df["mid"]) # “Does mid at time t predict future mid move?”

df = df.dropna()

"""
Step 2 (next upgrade)

Add 5s horizon

see if it improves Sharpe
check correlation between 1s and 5s (important)
Step 3 (final form)

Move to multi-horizon model

Option B — 1 model, 2 outputs (better long-term)

Multi-output regression:

X → [y_1s, y_5s]

expected_return = 0.7 * y_1s_pred + 0.3 * y_5s_pred

"""

# h1 = 10   # 1 second (100ms * 10)
# h5 = 50   # 5 seconds (100ms * 50)

# # -------------------------
# # future mids
# # -------------------------
# df["future_mid_1s"] = df["mid"].shift(-h1)
# df["future_mid_5s"] = df["mid"].shift(-h5)

# # -------------------------
# # log returns (targets)
# # -------------------------
# df["y_1s"] = np.log(df["future_mid_1s"] / df["mid"])
# df["y_5s"] = np.log(df["future_mid_5s"] / df["mid"])

# # -------------------------
# # cleanup
# # -------------------------
# df = df.dropna()

# features = [
#     "spread",
#     "order_imbalance",
#     "microprice"
# ]

features = [
    "microprice",
    "spread",
    "order_imbalance",
    "trade_imbalance",
    "inventory",
    "volatility",
    "queue_ahead_bid",
    "queue_ahead_ask",
    # "microprice" prev location
]

In [75]:
df["y"]

0       4.758140e-08
1       4.758140e-08
2       4.758140e-08
3       4.758140e-08
4       4.758140e-08
            ...     
7098    6.593545e-06
7099   -6.774125e-08
7100   -6.768370e-08
7101   -6.768479e-08
7102   -6.768817e-08
Name: y, Length: 7103, dtype: float64

In [76]:
split = int(len(df) * 0.8)

train = df.iloc[:split]
test = df.iloc[split:]

# Even better (next step later):

# walk-forward validation

In [77]:
"""
5. Train XGBoost
"""

import xgboost as xgb

X_train = train[features]
y_train = train["y"]

X_test = test[features]
y_test = test["y"]

model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
)

model.fit(X_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [78]:
print(df[[
    "order_imbalance",
    "trade_imbalance",
    "microprice",
    "spread"
]].corrwith(df["y"]))

order_imbalance    0.130554
trade_imbalance    0.091101
microprice        -0.024303
spread             0.004876
dtype: float64


In [80]:
print(df["y"].describe())
print(df["y"].rolling(1000).std().mean())

count    7.103000e+03
mean    -4.850815e-07
std      1.112119e-05
min     -3.136416e-04
25%     -5.689050e-08
50%      1.114935e-08
75%      4.759449e-08
max      1.629689e-04
Name: y, dtype: float64
1.0772512320490047e-05


In [81]:
"""
6. Evaluate properly (NOT R²)

In trading, R² is almost useless.

You want:

A) Direction accuracy

"""

pred = model.predict(X_test)

hit_rate = (np.sign(pred) == np.sign(y_test)).mean()

print("hit_rate:", hit_rate)

"""
B) Correlation (more important)

"""

corr = np.corrcoef(pred, y_test)[0,1]

print("corr", corr)

"""
C) PnL proxy (MOST important)

"""

pnl_proxy = (pred * y_test).sum()
print("pnl_proxy:", pnl_proxy)

hit_rate: 0.24278676988036593
corr nan
pnl_proxy: -6.322730509236283e-10


c:\Users\admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [ ]:
"""
7. Now connect it to your engine

This is where your system becomes real.

Replace:

expected_return = model.predict(features)

Then:

vol = state.get_vol()
k = k0 / (vol + 1e-6)

reservation = mid + k * expected_return

"""

In [ ]:
""""
Method 1: PnL grid search (best practical method)

You already have dataset → use it.

Pick best Sharpe / drawdown ratio.

Small k
passive MM
high fill rate
low directional exposure
Large k
aggressive alpha trading
fewer fills
more directional PnL
higher adverse selection risk

7. What professionals actually do (key insight)

Instead of just Sharpe, they optimize:

Utility=E[PnL]−λ⋅Var(PnL)−γ⋅Inventory Risk

8. One-line answer

To compute Sharpe in your replay:

simulate fills → compute equity curve → take periodic returns → compute mean/std → scale appropriately

To tune k
0
	​

:

run grid search of k
0
	​

 values and pick the one that maximizes Sharpe (and is stable across time, not just peak performance)

"""


for k0 in [0.1, 0.5, 1, 2, 5, 10]:
    simulate_strategy(k0)
    compute_sharpe()

In [82]:
for h in [1, 2, 5, 10, 20]:
    y = np.log(df["microprice"].shift(-h) / df["microprice"])
    print(h, y.std(), y.mean())

1 7.098980040562953e-06 -2.4476384889653536e-07
2 1.112771957433915e-05 -4.895966361222018e-07
5 1.9110010066758416e-05 -1.225447374166965e-06
10 2.8787921456930138e-05 -2.4708485401551777e-06
20 4.298044573383746e-05 -4.982277833748613e-06


In [ ]:
for h in [1, 2, 5, 10, 20]:
    y = np.log(df["mid"].shift(-h) / df["mid"])
    print(h, y.std(), y.mean())

"""
Key insight #1 — this is NORMAL microstructure scaling
The important pattern:
std increases with √time-ish behavior
mean is ~0 (tiny negative drift, basically noise)

This tells us:

✔ your price series is well-formed
✔ returns behave like a near-martingale
✔ no structural bug in midprice construction

So your dataset is NOT broken.

4. Key insight #2 — signal is extremely small

Let’s quantify it:

At h = 10:

std ≈ 2.9e-05 = 0.0029%

That is:

29 basis points volatility per 1-second horizon (roughly)

This is very low volatility microstructure noise, which is expected for:

tight crypto books
short aggregation windows
midprice sampling
5. Key insight #3 — why your ML failed earlier

Now this explains everything:

Your target is:

extremely low variance noise (~10⁻⁵ scale)

That implies:

signal-to-noise ratio is tiny
most features will have near-zero linear correlation
models will look like they are “not learning”

So:

❗ your earlier “bad model metrics” were NOT surprising
❗ they were mathematically expected


4. The key insight you should take away

This is the most important part:

❗ You are not predicting price movement

You are predicting:

very small deviations in a near-random walk

So:

raw return prediction is extremely weak
signal exists in structure, not direction
5. Why microprice matters now

You used:

microprice as anchor

That is actually correct because:

microprice = conditional expectation of short-term mid

So you're effectively trying to predict:

mid_t+h - micro_t


That difference is:

extremely small → but slightly biased by order flow

6. Intuition (important)

Think of it like this:

midprice → noisy fair price

microprice → “informed guess” of next tick

So your model is NOT predicting:

where price goes

but:

whether microprice is slightly wrong

7. Why martingale behavior is actually GOOD for you

If prices were predictable:

everyone would arbitrage it away instantly
MM strategies would fail differently

But martingale-like behavior means:

✔ spreads exist
✔ adverse selection is probabilistic (not deterministic)
✔ inventory risk can be priced

This is exactly what market making needs.

"""

1 7.1002053059306574e-06 -2.447800794757321e-07
2 1.112910952405938e-05 -4.896291013763795e-07
5 1.9111552092891522e-05 -1.2255285816915e-06
10 2.878965487141655e-05 -2.4710119528283015e-06
20 4.298195310305427e-05 -4.982608892376641e-06


In [ ]:
model_name = "alpha_model.json"

model_path = os.path.join(folder_path, model_name)

model.save_model(model_path)

In [ ]:
"""
What is still missing (important)

If you want this to be interview-elite, not just “cool project”, add:

1. Research layer (this is key)

You need results like:

“imbalance improves directional accuracy by X%”
“optimal horizon is 300-800ms”
“inventory-aware quoting reduces variance by X%”

Even simple graphs matter more than more code.

"""

In [ ]:
# # Notes

# quotes are not getting filled at all with quoting logic -> not competitive
# make spread adaptive

# 1. Your fill model is currently unrealistic

# This is the most important weakness.

# Right now:

# if price == state.bid_quote:

# This assumes:

# every trade at your price may fill you
# but ignores queue position

# In real HFT:

# queue position is EVERYTHING
# most passive orders never fill
# adverse selection dominates

# Right now your code is evolving into a real event-driven MM simulator, but the architecture is still “research notebook style” rather than “exchange engine style.” For HFT/quant interviews, the separation of concerns matters almost as much as the alpha logic.

# The strongest version of this project is:

# Market Data Layer
# Strategy Layer
# Execution / Simulation Layer
# (optional but very strong) Risk Layer

# That architecture immediately signals:

# systems thinking
# low-latency awareness
# production-style design
# extensibility
# understanding of real trading stacks

# Your current code already contains these layers conceptually — they’re just intermingled.

# to generate dataset for ML fill model, queue position is estimate heuristically based on level size and a position factor (e.g., 0.3). This is a simplification, but it allows you to create a feature that captures the idea of “how much liquidity is ahead of me at this price level?” which is crucial for fill probability estimation.

# Need backtesting ML layer to generate fill probability

# 🔥 Why this stops your infinite failure loop
# You were previously:
# restarting entire system → destroying continuity
# hoping for overlap event → statistically rare
# desyncing every attempt
# Now:
# single lifecycle
# deterministic timeout
# controlled sync window
# no recursive restart corruption
# 🧠 Final insight (important for your interview)

# If you say this in an interview, it signals senior-level thinking:

# “My initial bug was treating L2 sync as a stateless matching problem, but it is actually a stateful streaming alignment problem requiring a single-lifecycle reconciliation window, not repeated restart attempts.”

# my quoting prices were only up to 2 decimal places, but the exchange operates at 0.01 tick size, so I was effectively quoting at 0.01 increments but with a lot of rounding noise, which made my quotes non-competitive and rarely filled. By implementing a proper tick conversion and ensuring all prices are aligned to the exchange’s tick size, I can now quote more accurately and increase my chances of getting filled.

# 🔥 What a 1-tick spread actually implies

# At:

# Bid = 77799.99
# Ask = 77800.00

# This is:

# ultra-tight market
# extremely fast queue turnover
# heavy competition at top of book
# lots of passive liquidity stacking

# So:

# If you place:
# bid = 77799.99 → you join a massive queue
# ask = 77800.00 → you join another massive queue

# You are now competing with:

# market makers co-located on Binance infra
# algos reacting in milliseconds
# constant cancellations/replacements

# compute_queue_ahead with level * 0.3 is too static, i should model FIFO correctly, if not im always not getting filled.

# learnt that on depth is to update order book and queue position, while on trade is to check if i got filled and update inventory/cash. This separation is crucial for accurate state management and realistic simulation of market dynamics.

# place_quotes should not be called on depth as it might be over requoting, it should run independently on a timer or certain conditions to avoid overfitting to every market update and to simulate more realistic trading behavior.

# adverse selection is around 75% of fills

# will need to log in attribution everytime on_fill

# also might need a regime detection

# Got worse PNL when rounding then adjusting when generate_quotes, rather than adjusting then adjusting
# “Because in a discrete limit order book, rounding is not a cosmetic step — 
# it defines execution price priority and queue position. 
# Changing when discretization happens alters spread formation and fill sequencing, which dominates PnL more than the alpha signal itself.”

# realised execution latency matters, did not include this

# react dashboard, as rich terminal jittery as more and more metric put out, use react to run own metrics

# realized i need to reliably run the same datasets, with the same config for backtesting smoothly -> need for modular architecture that can ingest different configs -> use of manifest file

# currently execution step is driven by polling, not on market data. will change so it will not react to stale quotes, moved on market data to ASYNC

# during dataset testing, signal to check if microprice is a good indicator of mid is too noisy, corr was too small and dataset was too small. moved back to primitive mid prediction

# model is currently using fair and skew to compute microprice. 

# since microprice is a good fair value estimator of future mid, can we estimate the adjusted microprice better with a custom alpha signal model?

# we can also detect regimes using a classifier to tweak inputs for our alpha signal model

In [5]:
manifest = {
            
    # params
    "run_id": 0,
    "mode": "live",

    "models": {
        "edge_model": "mm-core",
        "regime_model": "",
        "fill_model": ""
    },

    "exchange": "binance",
    "instrument": "btcusdt",
    "tick_size": 0.01,
    "gamma": 0.1,
    "alpha_imb": 0.2,
    "alpha_flow": 0.05,

    # files
    "folder_path": "",
    "files": {
        "trades": "trades.parquet",
        "snapshots": "snapshots.parquet",
        "quotes": "quotes.parquet",
        "fills": "fills.parquet",
        "replay_events": {
            "events": "events.parquet",
            "orderbook_snapshot": "orderbook_snapshot.json"
        }
    },

    # dashboard config
    "server_config": {
        "host": "0.0.0.0",
        "port": 8000
    }
}

manifest_path = r"D:\OneDrive\Trading\Market Making\data\manifest_live.json"

# 3. write manifest
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)